# UTS — Data Mining

## Nomor 1 — Membaca Dataset
Membaca file titanic.csv dan menyimpannya dalam variabel dataset.

In [144]:
import pandas as pd 

dataset = pd.read_csv('titanic.csv')

print("Dataset Titanic:")
print(dataset)

print("\nJumlah baris:", dataset.shape[0])
print("Jumlah kolom:", dataset.shape[1])


Dataset Titanic:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 .

## Nomor 2 — Mengambil Kolom Data
Mengambil kolom Sex, Age, Pclass, Fare, dan Survived.
Age yang kosong tetap dipertahankan untuk diisi melalui klasifikasi 3-NN.

In [145]:
data = dataset[["Sex", "Age", "Pclass", "Fare", "Survived"]].copy()

print("Data yang digunakan:")
print(data)

print("\nJumlah missing values setiap kolom:")
print(data.isna().sum())

Data yang digunakan:
        Sex   Age  Pclass     Fare  Survived
0      male  22.0       3   7.2500         0
1    female  38.0       1  71.2833         1
2    female  26.0       3   7.9250         1
3    female  35.0       1  53.1000         1
4      male  35.0       3   8.0500         0
..      ...   ...     ...      ...       ...
886    male  27.0       2  13.0000         0
887  female  19.0       1  30.0000         1
888  female   NaN       3  23.4500         0
889    male  26.0       1  30.0000         1
890    male  32.0       3   7.7500         0

[891 rows x 5 columns]

Jumlah missing values setiap kolom:
Sex           0
Age         177
Pclass        0
Fare          0
Survived      0
dtype: int64


 ## Nomor 3 — Menyiapkan Fitur Training untuk Imputasi
Mengambil fitur Sex, Pclass, Fare, dan Survived dari baris
yang memiliki nilai Age. Sex dikonversi menjadi angka agar
dapat digunakan dalam normalisasi dan perhitungan jarak 3-NN.

In [146]:
train_data = data.loc[
    data["Age"].notna(),
    ["Sex", "Pclass", "Fare", "Survived"]
].copy()

print(train_data)

        Sex  Pclass     Fare  Survived
0      male       3   7.2500         0
1    female       1  71.2833         1
2    female       3   7.9250         1
3    female       1  53.1000         1
4      male       3   8.0500         0
..      ...     ...      ...       ...
885  female       3  29.1250         0
886    male       2  13.0000         0
887  female       1  30.0000         1
889    male       1  30.0000         1
890    male       3   7.7500         0

[714 rows x 4 columns]


## Nomor 4 — Mengambil label Age dari baris yang Age-nya terisi ##

In [147]:
train_label = data.loc[data['Age'].notna(), "Age"].copy()

print(train_label)

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
885    39.0
886    27.0
887    19.0
889    26.0
890    32.0
Name: Age, Length: 714, dtype: float64


## Nomor 5 — Mengambil fitur dari baris yang Age-nya kosong

In [148]:
test_data = data.loc[
    data["Age"].isna(),
    ["Sex", "Pclass", "Fare", "Survived"]
].copy()

print(test_data)

        Sex  Pclass     Fare  Survived
5      male       3   8.4583         0
17     male       2  13.0000         1
19   female       3   7.2250         1
26     male       3   7.2250         0
28   female       3   7.8792         1
..      ...     ...      ...       ...
859    male       3   7.2292         0
863  female       3  69.5500         0
868    male       3   9.5000         0
878    male       3   7.8958         0
888  female       3  23.4500         0

[177 rows x 4 columns]


## Nomor 6 — Normalisasi training dengan Min-Max 0–1

In [149]:
train_data["Sex"] = train_data["Sex"].replace({
    "male": 0,
    "female": 1
}).astype(int)

min_train = train_data.min()
max_train = train_data.max()

rentang_train = (max_train - min_train).replace(0, 1)
train_data = (train_data - min_train) / rentang_train

print("Minimum:")
print(min_train)

print("\nMaksimum:")
print(max_train)

print("\nHasil normalisasi training:")
print(train_data)

Minimum:
Sex         0.0
Pclass      1.0
Fare        0.0
Survived    0.0
dtype: float64

Maksimum:
Sex           1.0000
Pclass        3.0000
Fare        512.3292
Survived      1.0000
dtype: float64

Hasil normalisasi training:
     Sex  Pclass      Fare  Survived
0    0.0     1.0  0.014151       0.0
1    1.0     0.0  0.139136       1.0
2    1.0     1.0  0.015469       1.0
3    1.0     0.0  0.103644       1.0
4    0.0     1.0  0.015713       0.0
..   ...     ...       ...       ...
885  1.0     1.0  0.056848       0.0
886  0.0     0.5  0.025374       0.0
887  1.0     0.0  0.058556       1.0
889  0.0     0.0  0.058556       1.0
890  0.0     1.0  0.015127       0.0

[714 rows x 4 columns]


## Nomor 7 — Normalisasi test menggunakan minimum dan maksimum training

In [150]:
test_data["Sex"] = test_data["Sex"].replace({
    "male": 0,
    "female": 1
}).astype(int)

test_data = (test_data - min_train) / rentang_train

print(test_data)

     Sex  Pclass      Fare  Survived
5    0.0     1.0  0.016510       0.0
17   0.0     0.5  0.025374       1.0
19   1.0     1.0  0.014102       1.0
26   0.0     1.0  0.014102       0.0
28   1.0     1.0  0.015379       1.0
..   ...     ...       ...       ...
859  0.0     1.0  0.014110       0.0
863  1.0     1.0  0.135753       0.0
868  0.0     1.0  0.018543       0.0
878  0.0     1.0  0.015412       0.0
888  1.0     1.0  0.045771       0.0

[177 rows x 4 columns]


## Nomor 8 — Klasifikasi 3-NN untuk memprediksi Age

In [151]:
from sklearn.neighbors import KNeighborsClassifier

test_data = test_data.loc[:, train_data.columns]

model_imputasi = KNeighborsClassifier(n_neighbors=3)
model_imputasi.fit(train_data, train_label.astype(str))

class_result = model_imputasi.predict(test_data).astype(float)

print(class_result)

[21.   31.   13.   30.   19.   28.   58.   22.   20.   28.   20.   32.
 22.    2.   28.   30.   20.   28.   20.   22.   20.   20.   28.   21.
 31.   20.   51.   31.   32.   22.   26.    1.   22.   47.   24.   16.
 23.   31.    1.   51.   22.    1.   51.   28.   10.   21.   14.5   1.
 22.   18.   51.   21.   29.   22.   30.   47.   30.   27.   22.    1.
 17.   20.   17.    1.   31.   35.   28.    1.   45.   30.   19.   19.
 32.   13.   22.   24.   28.   21.   10.   28.   19.   30.   23.   28.
 22.   51.    1.   16.   16.   20.   35.   51.   20.   30.   21.   22.
 31.   30.   10.   16.   26.   36.   21.   28.   20.   24.   30.   15.
 19.   15.   31.   19.    0.67 18.   19.   51.   20.   23.   15.   22.
 14.5  26.   20.   21.   14.   30.   28.   28.   20.    1.   51.   21.
 38.   21.   32.   22.   28.   19.   28.   19.   35.   30.   23.   32.
 16.   20.   60.   32.   16.   30.   28.   28.   26.   19.   36.   24.
 30.   51.   21.   24.   51.   16.   29.   38.   29.    1.   25.   15.
 20.  

## Nomor 9 — Mengisi Age yang kosong dengan hasil prediksi

In [152]:
data.loc[test_data.index, "Age"] = class_result

print(data)
print("\nJumlah Age yang masih kosong:", data["Age"].isna().sum())

        Sex   Age  Pclass     Fare  Survived
0      male  22.0       3   7.2500         0
1    female  38.0       1  71.2833         1
2    female  26.0       3   7.9250         1
3    female  35.0       1  53.1000         1
4      male  35.0       3   8.0500         0
..      ...   ...     ...      ...       ...
886    male  27.0       2  13.0000         0
887  female  19.0       1  30.0000         1
888  female  10.0       3  23.4500         0
889    male  26.0       1  30.0000         1
890    male  32.0       3   7.7500         0

[891 rows x 5 columns]

Jumlah Age yang masih kosong: 0


## Nomor 10 — Membaca dataset test

In [153]:
test_dataset = pd.read_csv("titanic_test.csv")

print("Data test:")
print(test_dataset)

Data test:
     PassengerId  Pclass                                          Name  \
0            892       3                              Kelly, Mr. James   
1            893       3              Wilkes, Mrs. James (Ellen Needs)   
2            894       2                     Myles, Mr. Thomas Francis   
3            895       3                              Wirz, Mr. Albert   
4            896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)   
..           ...     ...                                           ...   
413         1305       3                            Spector, Mr. Woolf   
414         1306       1                  Oliva y Ocana, Dona. Fermina   
415         1307       3                  Saether, Mr. Simon Sivertsen   
416         1308       3                           Ware, Mr. Frederick   
417         1309       3                      Peter, Master. Michael J   

        Sex   Age  SibSp  Parch              Ticket      Fare Cabin Embarked  
0      male  34.5    

## Nomor 11 — Mengambil fitur training untuk memprediksi Survived

In [154]:
train_data = data[["Sex", "Age", "Pclass", "Fare"]].copy()

print("Data train:")
print(train_data)

Data train:
        Sex   Age  Pclass     Fare
0      male  22.0       3   7.2500
1    female  38.0       1  71.2833
2    female  26.0       3   7.9250
3    female  35.0       1  53.1000
4      male  35.0       3   8.0500
..      ...   ...     ...      ...
886    male  27.0       2  13.0000
887  female  19.0       1  30.0000
888  female  10.0       3  23.4500
889    male  26.0       1  30.0000
890    male  32.0       3   7.7500

[891 rows x 4 columns]


## Nomor 12 — Mengambil label Survived dari data training

In [155]:
train_label = data["Survived"].copy()

print("Label train:")
print(train_label)

Label train:
0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64


## Nomor 13 — Mengambil fitur test dan menghapus baris dengan missing values

In [156]:
test_data = test_dataset[["Sex", "Age", "Pclass", "Fare"]].copy()

test_data = test_data.dropna()

print("Data test:")
print(test_data)

Data test:
        Sex   Age  Pclass      Fare
0      male  34.5       3    7.8292
1    female  47.0       3    7.0000
2      male  62.0       2    9.6875
3      male  27.0       3    8.6625
4    female  22.0       3   12.2875
..      ...   ...     ...       ...
409  female   3.0       3   13.7750
411  female  37.0       1   90.0000
412  female  28.0       3    7.7750
414  female  39.0       1  108.9000
415    male  38.5       3    7.2500

[331 rows x 4 columns]


## Nomor 14 — Membaca label test sesuai baris yang tersisa

In [157]:
test_label_dataset = pd.read_csv("titanic_testlabel.csv")

test_label = test_dataset.loc[test_data.index, "PassengerId"].map(
    test_label_dataset.set_index("PassengerId")["Survived"]
)

print(test_label)

0      0
1      1
2      0
3      0
4      1
      ..
409    1
411    1
412    1
414    1
415    0
Name: PassengerId, Length: 331, dtype: int64


## Nomor 15 — Normalisasi training dengan Min-Max 0–1

In [158]:
train_data["Sex"] = train_data["Sex"].replace({
    "male": 0,
    "female": 1
}).astype(int)

min_train = train_data.min()
max_train = train_data.max()

rentang_train = (max_train - min_train).replace(0, 1)
train_data = (train_data - min_train) / rentang_train

print("Minimum:")
print(min_train)

print("\nMaksimum:")
print(max_train)

print("\nHasil normalisasi training:")
print(train_data)

Minimum:
Sex       0.00
Age       0.42
Pclass    1.00
Fare      0.00
dtype: float64

Maksimum:
Sex         1.0000
Age        80.0000
Pclass      3.0000
Fare      512.3292
dtype: float64

Hasil normalisasi training:
     Sex       Age  Pclass      Fare
0    0.0  0.271174     1.0  0.014151
1    1.0  0.472229     0.0  0.139136
2    1.0  0.321438     1.0  0.015469
3    1.0  0.434531     0.0  0.103644
4    0.0  0.434531     1.0  0.015713
..   ...       ...     ...       ...
886  0.0  0.334004     0.5  0.025374
887  1.0  0.233476     0.0  0.058556
888  1.0  0.120382     1.0  0.045771
889  0.0  0.321438     0.0  0.058556
890  0.0  0.396833     1.0  0.015127

[891 rows x 4 columns]


## Nomor 16 — Normalisasi test menggunakan minimum dan maksimum training

In [159]:
test_data["Sex"] = test_data["Sex"].replace({
    "male": 0,
    "female": 1
}).astype(int)

test_data = (test_data - min_train) / rentang_train

print(test_data)

     Sex       Age  Pclass      Fare
0    0.0  0.428248     1.0  0.015282
1    1.0  0.585323     1.0  0.013663
2    0.0  0.773813     0.5  0.018909
3    0.0  0.334004     1.0  0.016908
4    1.0  0.271174     1.0  0.023984
..   ...       ...     ...       ...
409  1.0  0.032420     1.0  0.026887
411  1.0  0.459663     0.0  0.175668
412  1.0  0.346569     1.0  0.015176
414  1.0  0.484795     0.0  0.212559
415  0.0  0.478512     1.0  0.014151

[331 rows x 4 columns]


## Nomor 17 — Klasifikasi Survived menggunakan 3-NN

In [160]:
from sklearn.neighbors import KNeighborsClassifier

test_data = test_data.loc[:, train_data.columns]

model = KNeighborsClassifier(n_neighbors=3)
model.fit(train_data, train_label)

class_result = model.predict(test_data)

print(class_result)

[0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1
 1 0 0 1 1 1 0 1 1 0 0 0 1 0 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 0 0 1 0 0 1 1 1
 0 1 1 1 1 0 0 0 1 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 1 0 0 0 0 0 1 1 0 0 1 0 0
 0 0 1 0 0 1 0 0 0 1 1 0 0 1 0 1 1 1 1 0 1 1 0 1 1 0 0 1 1 0 1 1 0 0 1 1 0
 1 0 0 0 1 0 1 0 1 0 0 1 0 1 0 1 0 1 0 0 0 1 0 1 0 1 1 0 1 1 1 0 0 0 0 1 0
 0 0 0 1 1 1 1 1 0 1 1 0 1 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 1 1 0 0 0 0 0 1 0
 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 1 1 1 0 0 1 0 0 0 0 1 1 0 1 0 0 0
 1 0 0 1 0 1 0 0 0 0 1 1 0 0 0 1 1 0 0 0 1 1 1 1 0 1 1 1 1 1 1 0 0 1 0 0 1
 1 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 1 0 0 0 1 1 0 1 0]


## Nomor 18 — Membandingkan prediksi dengan label test dan menghitung jumlah error

In [161]:
error = (class_result != test_label.to_numpy()).sum()

hasil = pd.DataFrame({
    "Aktual": test_label,
    "Prediksi": class_result
})
hasil["Salah"] = hasil["Aktual"] != hasil["Prediksi"]

print(hasil)
print("\nJumlah error:", error)

     Aktual  Prediksi  Salah
0         0         0  False
1         1         0   True
2         0         0  False
3         0         1   True
4         1         0   True
..      ...       ...    ...
409       1         1  False
411       1         1  False
412       1         0   True
414       1         1  False
415       0         0  False

[331 rows x 3 columns]

Jumlah error: 64


## Nomor 19 — Menghitung error ratio dalam persen

In [162]:
error_ratio = error / len(test_data) * 100

print("Jumlah error:", error)
print("Jumlah data test:", len(test_data))
print(f"Error ratio: {error_ratio:.2f}%")

Jumlah error: 64
Jumlah data test: 331
Error ratio: 19.34%
